In [1]:
!python -m pip install cityscapesscripts --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00


In [2]:
!csDownload

Cityscapes username or email address: ebrivera
Cityscapes password: 
Store credentials unencrypted in '/root/.local/share/cityscapesscripts/credentials.json' [y/N]: y


In [3]:
!csDownload leftImg8bit_trainvaltest.zip
!csDownload gtFine_trainvaltest.zip

Download progress:  98% 10.8G/11.0G [07:11<00:08, 26.9MB/s]
Download progress: 100% 241M/241M [00:10<00:00, 24.8MB/s]


In [4]:
# directory for data
!mkdir -p cityscapes_data

# extract from zip
!unzip leftImg8bit_trainvaltest.zip -d cityscapes_data/
!unzip gtFine_trainvaltest.zip -d cityscapes_data/

Streaming output truncated to the last 5000 lines.
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000117_000019_gtFine_color.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000114_000019_gtFine_color.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000434_000019_gtFine_labelIds.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000420_000019_gtFine_color.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000483_000019_gtFine_instanceIds.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000420_000019_gtFine_instanceIds.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000254_000019_gtFine_color.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000490_000019_gtFine_color.png  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000448_000019_gtFine_polygons.json  
  inflating: cityscapes_data/gtFine/test/berlin/berlin_000099_000019_gtFine_labelIds.png  
  inflating: cityscapes_data/gtFine/test/berlin

In [ ]:
!ls

cityscapes_data		 leftImg8bit_trainvaltest.zip
gtFine_trainvaltest.zip  sample_data


In [6]:
!ls cityscapes_data/


gtFine	leftImg8bit  license.txt  README


In [ ]:
# !ls cityscapes_data/gtFine/test/berlin

In [ ]:
# !ls cityscapes_data/leftImg8bit/test/berlin


In [5]:
!pip install tensorflow --quiet

In [ ]:
!ls cityscapes_data/leftImg8bit/train

aachen	cologne     erfurt   jena	      strasbourg  ulm
bochum	darmstadt   hamburg  krefeld	      stuttgart   weimar
bremen	dusseldorf  hanover  monchengladbach  tubingen	  zurich


In [ ]:
# !ls cityscapes_data/gtFine/test/berlin

In [7]:
# imports
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Input, Model
from tensorflow.keras.losses import Huber
from cityscapesscripts.helpers.annotation import Annotation
from cityscapesscripts.helpers.labels import labels, name2label, id2label, trainId2label, category2labels

In [8]:
# CONSTANTTS
CITYSCAPES_ROOT = 'cityscapes_data'
IMG_DIR = os.path.join(CITYSCAPES_ROOT, 'leftImg8bit')
GT_DIR = os.path.join(CITYSCAPES_ROOT, 'gtFine')

IMAGE_HEIGHT = 256
IMAGE_WIDTH = 512
IMAGE_CHANNELS = 3
BATCH_SIZE = 16

# for the ids in cityscape
VEHICLE_TRAIN_IDS = [13,14,15,31,33] # car, truck, bus, train, bicycle
NUM_DET_CLASSES = len(VEHICLE_TRAIN_IDS)

In [9]:
# pairing & basic load/resize
def get_file_pairs(split):
    """Return list of (image_path, label_path) for a given split."""
    pairs = []
    for city in sorted(os.listdir(os.path.join(IMG_DIR, split))):
        img_city = os.path.join(IMG_DIR, split, city) # path to images for the city
        gt_city = os.path.join(GT_DIR, split, city) # path to labels for the city
        for file_name in os.listdir(img_city):
            base = file_name.replace('_leftImg8bit.png', '')
            gt_fname = base + '_gtFine_labelIds.png'
            gt_path = os.path.join(gt_city, gt_fname) # expected gt filename
            if os.path.exists(gt_path):
                pairs.append((os.path.join(img_city, file_name), gt_path)) # append to apirs
    print(f"{split}: {len(pairs)} samples")
    return pairs

def load_image(path):
    """load, resize, then normalize"""
    img = Image.open(path).resize((IMAGE_WIDTH, IMAGE_HEIGHT), Image.BILINEAR) # linear interpolation
    np_array = np.array(img, dtype=np.float32) / 255.0
    return np_array

def load_label(path):
    """
        load, resize, then convert to trainIds
        the labels are actually in pngs w class ids bc of polygonal annotation

    """
    label = Image.open(path).resize((IMAGE_WIDTH, IMAGE_HEIGHT), Image.NEAREST) # these are harsh corners don't want linear interpolation
    np_array = np.array(label, dtype=np.int32)
    return np_array

def convert_label_to_trainids(label_arr):
    """maping cityscapes labels to trainids and blocking everything else"""
    out = np.full(label_arr.shape, 255, dtype=np.uint8)
    for L in labels:
        if 0 <= L.id and L.trainId != 255:
            out[label_arr == L.id] = L.trainId
    return out


In [10]:
# Quick test
train_pairs = get_file_pairs('train')
val_pairs =get_file_pairs('val')
test_pairs =get_file_pairs('test')

train: 2975 samples
val: 500 samples
test: 1525 samples


In [14]:
# prepping the detection
### THIS PROVED TO BE THE BIGGEST PROBLEM! BECUASE WE DIDN'T TAKE INTO ACCOUNT MORE THAN ONE OF A TID!!
def extract_largest_vehicle_bbox(gt_path):
    """
    return the largest blob of pixels
    """
    mask = convert_label_to_trainids(load_label(gt_path))
    candidates = []
    for tid in VEHICLE_TRAIN_IDS:
        ys, xs = np.nonzero(mask == tid)
        if ys.size:
            y1, y2 = ys.min(), ys.max()
            x1, x2 = xs.min(), xs.max()
            area = (x2-x1)*(y2-y1)
            candidates.append((area, (x1, y1, x2, y2)))
    return max(candidates)[1] if candidates else None


In [15]:
def prepare_detection_data(pairs, max_samples=None):
    """
    pairs: list of (image_path, gt_path)
    returns: X, y_c (class idx), y_b (normalized boxes)
    """
    if max_samples:
        pairs = pairs[:max_samples]
    N = len(pairs)
    X= np.zeros((N, IMAGE_HEIGHT, IMAGE_WIDTH, 3), dtype=np.float32)
    y_c = np.zeros((N,), dtype=np.int32)
    y_b = np.zeros((N, 4), dtype=np.float32)

    idx = 0
    for img_p, gt_p in pairs:

        img = load_image(img_p)

        # get the largest BOUNDING BOX BLOB from the GT file
        box = extract_largest_vehicle_bbox(gt_p)
        if box is None:
            continue
        x1, y1, x2, y2 = box

        # load the mask once, sample its center to get the trainId
        mask = convert_label_to_trainids(load_label(gt_p))
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        trainid = int(mask[cy, cx])
        # turn it into its class id from the vehicles
        if trainid not in VEHICLE_TRAIN_IDS:
            continue
        cls_i = VEHICLE_TRAIN_IDS.index(trainid)

        X[idx] = img
        y_c[idx] = cls_i
        w, h = x2 - x1, y2 - y1
        y_b[idx] = [x1/IMAGE_WIDTH, y1/IMAGE_HEIGHT, w/IMAGE_WIDTH, h/IMAGE_HEIGHT]
        idx += 1

    #trim off any leftover slots, didn't have trainid in vehicle trains
    X = X[:idx]
    y_c = y_c[:idx]
    y_b = y_b[:idx]

    return X, y_c, y_b


In [16]:
# create the tensorflow datasets
train_pairs = get_file_pairs('train')
val_pairs= get_file_pairs('val')
X_tr, y_tr_c, y_tr_b = prepare_detection_data(train_pairs, max_samples=2000)
X_va, y_va_c, y_va_b = prepare_detection_data(val_pairs,   max_samples= 500)

# need to train based on class and bounding box
train_ds = tf.data.Dataset.from_tensor_slices(
    (X_tr, {'class_output': y_tr_c, 'bbox_output': y_tr_b})
).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE) # with a shuffle + batching and prefetching which is something i learned while coding my own models

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_va, {'class_output': y_va_c, 'bbox_output': y_va_b})
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train: 2975 samples
val: 500 samples


In [17]:
def giou_loss(y_true, y_pred):
    """
    https://giou.stanford.edu/
    returns (1 - GIoU) per sample.
    """
    # unpack
    x0_t, y0_t, w_t, h_t = tf.split(y_true, 4, -1)
    x0_p, y0_p, w_p, h_p = tf.split(y_pred, 4, -1)
    x1_t, y1_t = x0_t, y0_t
    x2_t, y2_t = x0_t + w_t, y0_t + h_t
    x1_p, y1_p = x0_p, y0_p
    x2_p, y2_p = x0_p + w_p, y0_p + h_p

    # intersection
    xi1 = tf.maximum(x1_t, x1_p); yi1 = tf.maximum(y1_t, y1_p)
    xi2 = tf.minimum(x2_t, x2_p); yi2 = tf.minimum(y2_t, y2_p)
    iw = tf.maximum(0.0, xi2 - xi1)
    ih = tf.maximum(0.0, yi2 - yi1)
    inter = iw * ih

    # union
    area_t = (x2_t - x1_t) * (y2_t - y1_t)
    area_p = (x2_p - x1_p) * (y2_p - y1_p)
    union = area_t + area_p - inter

    # IoU
    iou = inter / (union + 1e-7) # 1e-7 to make sure it doesn't nan

    # smallest enclosing box
    xc1 = tf.minimum(x1_t, x1_p); yc1 = tf.minimum(y1_t, y1_p)
    xc2 = tf.maximum(x2_t, x2_p); yc2 = tf.maximum(y2_t, y2_p)
    cw = tf.maximum(0.0, xc2 - xc1)
    ch = tf.maximum(0.0, yc2 - yc1)
    area_c = cw * ch

    giou = iou - (area_c - union) / (area_c + 1e-7)
    return 1.0 - giou # shape [batch,1]

In [18]:

def build_base_model():
    inp = Input((IMAGE_HEIGHT,IMAGE_WIDTH,3), name='input_image')
    x = layers.Conv2D(32,3,activation='relu',padding='same')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
    x = layers.MaxPooling2D()(x)
    feat= layers.Conv2D(128,3,activation='relu',padding='same')(x)
    x = layers.GlobalAveragePooling2D()(feat)

    # small regression head
    bx = layers.Dense(128, activation='relu')(x)
    box = layers.Dense(4, activation='linear', name='bbox_output')(bx)

    cls = layers.Dense(NUM_DET_CLASSES, activation='softmax', name='class_output')(x)
    return Model(inp, [cls, box], name='BaseDetector')

In [19]:
def build_bn_model():
    inp = keras.Input((IMAGE_HEIGHT, IMAGE_WIDTH, 3), name='input_image')
    # shared convs (unchanged)
    x = layers.Conv2D(32,3,padding='same', activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64,3,padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128,3,padding='same', activation='relu')(x)
    feat = layers.GlobalAveragePooling2D()(x)

    # seperate head for the bbox
    b = layers.Dense(256, activation='relu')(feat)
    b = layers.Dense(128, activation='relu')(b)
    box_output = layers.Dense(4, activation='linear', name='bbox_output')(b)

    # seperate head for the classification
    c = layers.Dropout(0.5)(feat)
    cls_output = layers.Dense(NUM_DET_CLASSES,
                              activation='softmax',
                              name='class_output')(c)

    return keras.Model(inp, [cls_output, box_output], name='BNDetector')


In [20]:
def build_deeper_model():
    inp = Input((IMAGE_HEIGHT, IMAGE_WIDTH, 3), name='input_image')

    # shared cnn
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    feat = layers.GlobalAveragePooling2D()(x)

    # seperating head for bbox
    b = layers.Dense(256, activation='relu')(feat)
    b = layers.Dense(128, activation='relu')(b)
    box_output = layers.Dense(4, activation='linear', name='bbox_output')(b)

    # seperate head for classification
    c = layers.Dropout(0.5)(feat)
    class_output = layers.Dense(
        NUM_DET_CLASSES,
        activation='softmax',
        name='class_output'
    )(c)

    return Model(inp, [class_output, box_output], name='DeeperDetector')

In [21]:
def compile_and_train(model, train_ds, val_ds, name, epochs=20, patience=3):
    # pure giou loss
    bbox_loss = giou_loss

    model.compile(
      optimizer='adam',
      loss={
        'class_output':'sparse_categorical_crossentropy',
        'bbox_output': bbox_loss
      },
      loss_weights={
          'class_output': 1.0,
          'bbox_output': 5.0 # increase the weight of bbox because that is more important for us (classification doesn't requrie as much)
        },
      metrics={'class_output':'accuracy'}
    )
    print(f"\nTraining {name}")
    return model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[
          keras.callbacks.EarlyStopping(
            monitor='val_bbox_output_loss', # again more focused on bbox
            mode='min',
            patience=patience,
            restore_best_weights=True
          )
        ]
    )



In [22]:
base_model = build_base_model()
base_hist = compile_and_train(base_model, train_ds, val_ds, 'BaseDetector')


Training BaseDetector
Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 188ms/step - bbox_output_loss: 1.2212 - class_output_accuracy: 0.8243 - class_output_loss: 1.1976 - loss: 7.3041 - val_bbox_output_loss: 0.8987 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5720 - val_loss: 5.0741
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - bbox_output_loss: 0.8206 - class_output_accuracy: 0.9146 - class_output_loss: 0.4328 - loss: 4.5359 - val_bbox_output_loss: 0.8359 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5545 - val_loss: 4.7379
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - bbox_output_loss: 0.8135 - class_output_accuracy: 0.9096 - class_output_loss: 0.4123 - loss: 4.4799 - val_bbox_output_loss: 0.8118 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5367 - val_loss: 4.5896
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - bbox_output_loss: 0.8065 - class_output_accuracy: 0.9124 - class_output_loss: 0.3988 - loss: 4.4313 - val_bbo

In [23]:
base_model.summary()

Model: "BaseDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 512,  │        896 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 256,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 256,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 128,   │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 128,   │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 5)         │        645 │ global_average_p… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │        516 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 332,765 (1.27 MB)

 Trainable params: 110,921 (433.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 221,844 (866.58 KB)

In [24]:
bn_model = build_bn_model()
bn_hist = compile_and_train(bn_model, train_ds, val_ds, 'BNDetector')


Training BNDetector
Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 216ms/step - bbox_output_loss: 1.1118 - class_output_accuracy: 0.8203 - class_output_loss: 0.9817 - loss: 6.5408 - val_bbox_output_loss: 0.8656 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.6224 - val_loss: 4.9548
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - bbox_output_loss: 0.8193 - class_output_accuracy: 0.9040 - class_output_loss: 0.5023 - loss: 4.5986 - val_bbox_output_loss: 0.8169 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5557 - val_loss: 4.6426
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - bbox_output_loss: 0.8053 - class_output_accuracy: 0.9036 - class_output_loss: 0.4612 - loss: 4.4878 - val_bbox_output_loss: 0.8303 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5459 - val_loss: 4.7019
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - bbox_output_loss: 0.7973 - class_output_accuracy: 0.9212 - class_output_loss: 0.3695 - loss: 4.3563 - val_bbox_

In [25]:
bn_model.summary()

Model: "BNDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 512,  │        896 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 128, 256,  │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 256,  │     18,496 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 128,   │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 128,   │     73,856 │ max_pooling2d_3[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_5[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     33,024 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 5)         │        645 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │        516 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 480,989 (1.83 MB)

 Trainable params: 160,329 (626.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 320,660 (1.22 MB)

In [38]:
deep_model = build_deeper_model()
deep_hist = compile_and_train(deep_model, train_ds, val_ds, 'DeeperDetector')


Training DeeperDetector
Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 103ms/step - bbox_output_loss: 1.1184 - class_output_accuracy: 0.8273 - class_output_loss: 0.8923 - loss: 6.4846 - val_bbox_output_loss: 1.0207 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5859 - val_loss: 5.6994
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - bbox_output_loss: 0.8701 - class_output_accuracy: 0.9172 - class_output_loss: 0.4173 - loss: 4.7681 - val_bbox_output_loss: 0.8299 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5467 - val_loss: 4.6918
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - bbox_output_loss: 0.8066 - class_output_accuracy: 0.8995 - class_output_loss: 0.4791 - loss: 4.5123 - val_bbox_output_loss: 0.8117 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5405 - val_loss: 4.5973
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - bbox_output_loss: 0.7829 - class_output_accuracy: 0.9153 - class_output_loss: 0.3948 - loss: 4.3093 - val_b

In [39]:
deep_model.summary()

Model: "DeeperDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 256, 512,  │        896 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_13    │ (None, 128, 256,  │          0 │ conv2d_18[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 128, 256,  │     18,496 │ max_pooling2d_13… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 64, 128,   │          0 │ conv2d_19[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 64, 128,   │     73,856 │ max_pooling2d_14… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 32, 64,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 64,    │    295,168 │ max_pooling2d_15… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv2d_21[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 128)       │     32,896 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 5)         │      1,285 │ dropout_4[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │        516 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,466,717 (5.60 MB)

 Trainable params: 488,905 (1.87 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 977,812 (3.73 MB)

In [40]:
def Build_AugDeeperDetector():
    inp = Input((IMAGE_HEIGHT, IMAGE_WIDTH, 3), name='input_image')

    # augmentation randomly
    x = layers.RandomFlip('horizontal')(inp)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1, 0.1)(x)

    # shared cnn
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    feat = layers.GlobalAveragePooling2D()(x)

    # this is again for the bbox
    b = layers.Dense(256, activation='relu')(feat)
    b = layers.Dense(128, activation='relu')(b)
    box_output = layers.Dense(4, activation='linear', name='bbox_output')(b)

    # classification
    c = layers.Dropout(0.5)(feat)
    class_output = layers.Dense(
        NUM_DET_CLASSES,
        activation='softmax',
        name='class_output'
    )(c)

    return Model(inp, [class_output, box_output], name='AugDeeperDetector')


In [41]:
aug_deep_model = Build_AugDeeperDetector()
aug_deep_hist = compile_and_train(aug_deep_model, train_ds, val_ds, 'AugDeeperDetector')


Training AugDeeperDetector
Epoch 1/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - bbox_output_loss: 1.1165 - class_output_accuracy: 0.8834 - class_output_loss: 0.7596 - loss: 6.3423 - val_bbox_output_loss: 0.9086 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5822 - val_loss: 5.1292
Epoch 2/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - bbox_output_loss: 0.8478 - class_output_accuracy: 0.9047 - class_output_loss: 0.4713 - loss: 4.7102 - val_bbox_output_loss: 0.8057 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5566 - val_loss: 4.5856
Epoch 3/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - bbox_output_loss: 0.8059 - class_output_accuracy: 0.9180 - class_output_loss: 0.3897 - loss: 4.4193 - val_bbox_output_loss: 0.8117 - val_class_output_accuracy: 0.8627 - val_class_output_loss: 0.5287 - val_loss: 4.5898
Epoch 4/20
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - bbox_output_loss: 0.7973 - class_output_accuracy: 0.9274 - class_output_loss: 0.3617 - loss: 4.3483 - val_

In [30]:
aug_deep_model.summary()

Model: "AugDeeperDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 256, 512,  │          0 │ input_image[0][0] │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 256, 512,  │          0 │ random_flip[0][0] │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom         │ (None, 256, 512,  │          0 │ random_rotation[… │
│ (RandomZoom)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 256, 512,  │        896 │ random_zoom[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 128, 256,  │          0 │ conv2d_10[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 128, 256,  │     18,496 │ max_pooling2d_7[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 64, 128,   │          0 │ conv2d_11[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 64, 128,   │     73,856 │ max_pooling2d_8[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 32, 64,    │          0 │ conv2d_12[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 32, 64,    │    295,168 │ max_pooling2d_9[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ conv2d_13[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │     32,896 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 5)         │      1,285 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │        516 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,466,717 (5.60 MB)

 Trainable params: 488,905 (1.87 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 977,812 (3.73 MB)

In [31]:
def compute_iou(boxA, boxB):
    # https://giou.stanford.edu/
    xA1,yA1,xA2,yA2 = boxA
    xB1,yB1,xB2,yB2 = boxB
    xi1,yi1 = max(xA1,xB1), max(yA1,yB1)
    xi2,yi2 = min(xA2,xB2), min(yA2,yB2)
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    areaA = (xA2-xA1) * (yA2-yA1)
    areaB = (xB2-xB1) * (yB2-yB1)
    return inter / (areaA + areaB - inter + 1e-8)

In [45]:
def eval_model_iou(model, val_imgs):
    ious = []
    for img_p in val_imgs:
        gt_p = (img_p
                .replace('/leftImg8bit/','/gtFine/')
                .replace('_leftImg8bit.png','_gtFine_labelIds.png'))
        gt_box = extract_largest_vehicle_bbox(gt_p)
        if gt_box is None:
            continue

        img = load_image(img_p)[None,...]
        raw = model.predict(img, verbose=0)[1]
        x0,y0,w,h = raw[0] * np.array([IMAGE_WIDTH,
                                       IMAGE_HEIGHT,
                                       IMAGE_WIDTH,
                                       IMAGE_HEIGHT])
        pred_box = [x0, y0, x0 + w, y0 + h]
        ious.append(compute_iou(pred_box, gt_box))

    mean_iou = np.mean(ious)
    frac = np.mean([i>=0.5 for i in ious])
    print(f"{model.name}: mean IoU = {mean_iou:.3f}, IoU>=0.5 = {frac:.3f}") # second part tells us how many were at least 50% correct


In [46]:
from glob import glob
val_imgs = glob(os.path.join(IMG_DIR,'val','*','*.png'))[:20]

eval_model_iou(base_model, val_imgs)
eval_model_iou(bn_model, val_imgs)
eval_model_iou(deep_model, val_imgs)
eval_model_iou(aug_deep_model, val_imgs)


BaseDetector: mean IoU = 0.304, IoU>=0.5 = 0.312
BNDetector: mean IoU = 0.302, IoU>=0.5 = 0.312
DeeperDetector: mean IoU = 0.298, IoU>=0.5 = 0.312
AugDeeperDetector: mean IoU = 0.285, IoU>=0.5 = 0.312


In [35]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.0 MB/s eta 0:00:00


In [48]:
from ultralytics import YOLO
from glob import glob
import os
import numpy as np

yolo = YOLO('yolov8n.pt')
VEHICLE_COCO_IDS = [2, 3, 5, 7] # this is from yolo's pretrained models


val_imgs = glob(os.path.join(IMG_DIR, 'val', '*', '*.png'))[:20]
ious = []

for img_p in val_imgs:
    # build boxes
    gt_p = img_p.replace('/leftImg8bit/','/gtFine/').replace('_leftImg8bit.png','_gtFine_labelIds.png')
    gt_box = extract_largest_vehicle_bbox(gt_p)
    if gt_box is None:
        continue

    # run yolo
    res = yolo.predict(source=img_p,
                       imgsz=(IMAGE_HEIGHT, IMAGE_WIDTH),
                       verbose=False)[0]

    # unpack
    orig_h, orig_w = res.orig_shape

    # unpack but need cpu bc numpy doesn't work on gpu
    boxes = res.boxes.xyxy.cpu().numpy()
    clsids = res.boxes.cls.cpu().numpy().astype(int)

    # again mask out everything but the vehicles
    mask = [c in VEHICLE_COCO_IDS for c in clsids]
    boxes = boxes[mask]
    if len(boxes)==0:
        continue
    scale_w = IMAGE_WIDTH / orig_w
    scale_h = IMAGE_HEIGHT / orig_h
    boxes[:, [0,2]] *= scale_w
    boxes[:, [1,3]] *= scale_h

    areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    pred= boxes[np.argmax(areas)] # get largest prediction box bc yolo gets all of them

    # get the iou
    ious.append(compute_iou(pred, gt_box))

print(f"\nYOLOv8-n baseline mean IoU: {np.mean(ious):.3f}, IoU>=0.5: {np.mean([i>0.5 for i in ious]):.3f}")



YOLOv8-n baseline   mean IoU: 0.339,  IoU>=0.5: 0.167
